"Copernicus" is not the name of a single satellite or instrument. 
Copernicus is the name of the Space Programme run by the European Union.

Just like Apple makes both iPhones (cameras) and MacBooks (computers), the Copernicus program runs completely different space missions that do completely different jobs.Here is exactly what those two IDs mean:1. The Optical Image (COPERNICUS/S2_SR_HARMONIZED)The Instrument: 

This data comes from the Sentinel-2 satellites.How it works: These are optical satellites flying in orbit right now with camera lenses. They capture sunlight reflecting off the earth to give you RGB, NIR, and SWIR bands.Crucial Detail: They cannot see through clouds and they do not measure elevation.2. 

The DEM Elevation Data (COPERNICUS/DEM/30m)The Instrument: This data comes primarily from a radar mission called TanDEM-X, which was a pair of specialized radar satellites flying together years ago.

How it works: They bounced radar waves off the ground to map the physical height of mountains and valleys.Crucial Detail: The European Union bought this global elevation map and published it under their "Copernicus" banner so everyone could use it.

The Breakthrough Dataset (Hugonnet et al. 2021)For a truly accurate ML model tracking mass change, the global standard dataset for time-series glacier thinning is the Hugonnet Global Glacier Change dataset. It maps out the exact rate of glacier elevation loss over the HKH range in 5-year blocks (2000–2004, 2005–2009, 2010–2014, 2015–2019).You can use their public open-source maps to mathematically subtract or add height to your baseline DEM based on the year you are inferencing!

Why this approach is safer:Downloading the data as one single 7-band block and slicing it locally via full_tensor[0:6] and full_tensor[6:7] prevents bugs. If you attempted to download the optical array and the DEM array as two completely isolated requests from Earth Engine, any minor processing or network roundoff variation could result in one array having dimensions like (6, 120, 120) while the other returned (1, 120, 121). Your model concatenation layer would throw an immediate shape mismatch crash. Slicing a single array guarantees identical H and W dimensions.

User Selects Year (e.g., 2026) 
       
       │
       ▼

[FastAPI Backend] 

Runs `get_consistent_4d_image(aoi, 2026)` ──► Downloads 7-Band 30m Grid via computePixels()

       │
       ▼

[PyTorch Model Inference]

Processes Optical Bands ──► Generates Binary Segmentation Mask Array (H x W)

       │
       ▼

[Output Translation]

1. Extracts Raw 'DEM' Array ──► Flat JSON array of raw elevations (Physical 3D Terrain Geometry)

2. Converts Mask via OpenCV ──► See-through PNG sticker overlay (Draped Glacier Color Layer)

       │
       ▼

[Frontend 4D Dashboard]

CesiumJS / Three.js updates the screen instantly, displaying the shrinking/growing boundaries.


1. How GEE computes everything automatically behind FastAPIWhen your FastAPI code runs import ee and initializes, it connects to Google’s Cloud servers via an API token.When your code executes get_consistent_4d_image, no image data is processed on your local FastAPI server, and nothing is downloaded yet.

Instead, every time you chain commands like .filterBounds(), .sort(), .multiply(), or .addBands(), the ee Python library is just translating those commands into a nested JSON dictionary recipe.For example, it creates a text-based instruction that says: 

"Find Landsat image X, multiply it by 0.0000275, find SRTM elevation data, match its grid, stack them together, and clip it to these coordinates."Your FastAPI server holds onto this lightweight text recipe inside the optical_image variable.

2. How the 6 Satellite Bands and 1 Elevation Band bind togetherSatellite data (Landsat) and Elevation data (SRTM) come from two completely different physical spacecraft instruments, not one.Google Earth Engine allows you to virtually "glue" them together into a single packet using the .addBands() function.



[Landsat Recipe Layer]       [SRTM Elevation Recipe Layer]
(Blue, Green, Red,           (DEM)
 NIR, SWIR1, SWIR2)             │
         │                      │  <- reprojected to match Landsat grid
         └──────────┬───────────┘
                    ▼
          [.addBands() Merges Them]
                    ▼
     [Final 7-Band Combined Recipe Layer]
Before binding them, your code runs .reproject(). 

This forces the SRTM elevation grid map to stretch, align, and perfectly match the exact pixel layout and coordinate grid of your Landsat satellite image.3. How fetch_as_numpy actually triggers the downloadUp until this point, you only have a recipe. The actual data generation and download happen in a single handshake inside fetch_as_numpy:

The Request (image.getDownloadURL): Your FastAPI server sends that entire JSON recipe to the Google Earth Engine cloud servers.The Cloud Execution: Google's servers read the recipe. Their massive server farm pulls the requested Landsat image and SRTM image from their hard drives, aligns them, clips the exact Area of Interest (aoi), renders a 512 × 512 pixel matrix, and bundles it into a raw binary GeoTIFF file containing all 7 bands.The Transfer (requests.get(url)): GEE gives your FastAPI server a temporary cloud link. 

Your FastAPI server calls requests.get(url) to download that finished binary file directly into your server's RAM memory (io.BytesIO).The Parsing (rasterio.open): Rasterio opens those raw bytes in RAM and decodes the image structure, turning it into a 3D NumPy array of shape (7, 512, 512) that your Prithvi model can immediately read.

The 3 Core Domain Shift ProblemsShift Type

The ProblemWhat Happens at Inference (2024)Sensor ShiftLILA uses Landsat 7 ETM+. 2013+ uses Landsat 8/9 OLI.

Band spectral response functions (SRFs) and wavelengths differ slightly between ETM+ and OLI.Atmospheric ShiftDifferent years have different snow cover, atmospheric haze, and sun angles.

The model misinterprets seasonal snow or high-altitude clouds as glacier ice.Sensor Hardware DefectLandsat 7 has scan-line corrector (SLC) gaps.The model learns LILA's black SLC stripe artifacts as structural features.

How to Fix Generalization (Without Labeling New Data)You solve domain generalization through pre-processing, model architecture, and runtime normalization, not by manually labeling 15 years of images.

1. Sensor & Radiometric Harmonization (GEE Side)Never feed raw Digital Numbers (DN) or raw Top-Of-Atmosphere (TOA) imagery to your model.Use Harmonized Landsat Data: In GEE, use Surface Reflectance datasets (LANDSAT/LE07/C02/T1_L2 for L7 and LANDSAT/LC08/C02/T1_L2 / LC09 for L8/L9).Cross-Calibration: Apply ETM+-to-OLI band regression transformations (or use HLS / Harmonized Landsat Sentinel logic) inside your GEE pipeline to align reflectance values across satellite generations.

2. Pre-trained Foundation Weights (Prithvi ViT)Prithvi was pre-trained by NASA/IBM on 4.2 million global HLS time-series patches across different sensors and years. Its encoder already knows what Earth looks like under different atmospheric conditions. Fine-tune only the decoder and top ViT layers with a low learning rate so you don't destroy its pre-trained invariant representations.

3. Heavy Domain Augmentation During TrainingWhen training on your 10 shards, apply aggressive spectral augmentations using Albumentation or Kornia:ColorJitter / Random Brightness & Contrast: Simulates seasonal lighting/sun angle shifts.Channel Shuffling / Gaussian Noise: Prevents over-reliance on exact L7 spectral intensity ratios.Random Cloud/Haze Injection: Forces the network to rely on DEM terrain features (slope/elevation) alongside optical bands.

4. Test-Time Normalization (Inference Side)Z-Score Scaling per Tile: Normalize each incoming 2024 tile using its own per-channel mean and standard deviation:$$\hat{X}_{c} = \frac{X_{c} - \mu_{c}}{\sigma_{c}}$$This strips away global brightness offsets caused by year-to-year atmospheric variations.